In [ ]:
import json
import argparse
import numpy as np
import onnxruntime as rt
import pandas as pd
from sklearn.impute import SimpleImputer
import os

# Set a seed for reproducibility, as requested by the rules.
np.random.seed(42)

# --- CONFIGURATION ---
# This section should match the models you trained.
MODEL_DIR = "trained_silo_models_v1" # The directory where you saved your models
PREDICTOR_CONFIG = {
    'DL': {
        'onnx_lon_path': os.path.join(MODEL_DIR, 'pipeline_lon_DL.onnx'),
        'onnx_lat_path': os.path.join(MODEL_DIR, 'pipeline_lat_DL.onnx'),
        'feature_cols': [ # Must be the exact list used for training
            'NR_UE_PCI_0', 'NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0',
            'NR_UE_Pathloss_DL_0', 'NR_UE_Nbr_RSRQ_0', 'NR_UE_Nbr_RSRP_0',
            'NR_UE_Nbr_PCI_0', 'NR_UE_Modulation_Avg_DL_0', 'NR_UE_Timing_Advance'
        ],
        'fusion_weight': 0.0 # We will fill this from our test results
    },
    'UL': {
        'onnx_lon_path': os.path.join(MODEL_DIR, 'pipeline_lon_UL.onnx'),
        'onnx_lat_path': os.path.join(MODEL_DIR, 'pipeline_lat_UL.onnx'),
        'feature_cols': [
            'NR_UE_PCI_0', 'NR_UE_RSRP_0', 'NR_UE_RSRQ_0', 'NR_UE_SINR_0',
            'NR_UE_Pathloss_UL_0', 'NR_UE_Nbr_RSRQ_0', 'NR_UE_Nbr_RSRP_0',
            'NR_UE_Nbr_PCI_0', 'NR_UE_Modulation_Avg_UL_0', 'NR_UE_Timing_Advance',
            'NR_UE_Power_Tx_PUSCH_0'
        ],
        'fusion_weight': 0.0
    },
    'Scanner': {
        'onnx_lon_path': os.path.join(MODEL_DIR, 'pipeline_lon_Scanner.onnx'),
        'onnx_lat_path': os.path.join(MODEL_DIR, 'pipeline_lat_Scanner.onnx'),
        'feature_cols': [ # From your scanner training
            'Scanner_Signal_Strength', 'Scanner_Band', 'Scanner_Noise_Floor'
        ],
        'fusion_weight': 0.0
    }
}

# IMPORTANT: Manually set these weights based on your training script's output
# Example: if DL median error was 5.2m, its weight is 1/5.2 = 0.1923
PREDICTOR_CONFIG['DL']['fusion_weight']      = 0.1923 # 1 / 5.2m error
PREDICTOR_CONFIG['UL']['fusion_weight']      = 0.1234 # 1 / 8.1m error
PREDICTOR_CONFIG['Scanner']['fusion_weight'] = 0.2500 # 1 / 4.0m error

# --- HELPER FUNCTIONS ---

def preprocess_input_data(data_dict, feature_cols):
    """
    Takes a dictionary of measurements, converts it to a DataFrame,
    and handles preprocessing like imputation. This must be embedded in the code.
    """
    # Create a DataFrame from the single row of data
    df = pd.DataFrame([data_dict])
    
    # Ensure all required feature columns are present, adding missing ones as NaN
    for col in feature_cols:
        if col not in df.columns:
            df[col] = np.nan
            
    # Select features in the correct order
    df = df[feature_cols]
    
    # Impute any missing values (e.g., if the source provided only some features)
    # Using 'mean' is simple; could use a pre-fitted imputer for better results
    imputer = SimpleImputer(strategy='mean')
    
    # The imputer needs to be fitted on something, fitting on the data itself is a simple approach for a single prediction
    imputed_data = imputer.fit_transform(df)
    
    return imputed_data.astype(np.float32)


def predict_from_source(input_data, config):
    """
    Loads ONNX models and performs prediction for a single data source.
    """
    try:
        # Load ONNX models using the InferenceSession
        lon_session = rt.InferenceSession(config['onnx_lon_path'])
        lat_session = rt.InferenceSession(config['onnx_lat_path'])
        
        lon_input_name = lon_session.get_inputs()[0].name
        lat_input_name = lat_session.get_inputs()[0].name
        
        # Preprocess the raw data
        processed_data = preprocess_input_data(input_data, config['feature_cols'])
        
        # Predict
        pred_lon = lon_session.run(None, {lon_input_name: processed_data})[0][0][0]
        pred_lat = lat_session.run(None, {lat_input_name: processed_data})[0][0][0]
        
        return pred_lon, pred_lat
        
    except FileNotFoundError:
        # This allows the code to continue if a model file is missing
        print(f"Warning: Model files not found for a source. Skipping prediction.")
        return None, None
    except Exception as e:
        print(f"An error occurred during prediction: {e}")
        return None, None

# --- MAIN EXECUTION LOGIC ---

if __name__ == "__main__":
    
    # 1. Setup to read command-line arguments like --input test.json
    parser = argparse.ArgumentParser(description="Predict coordinates from sensor data.")
    parser.add_argument('--input', type=str, required=True, help="Path to the input JSON file.")
    args = parser.parse_args()

    # 2. Load the input data from the provided JSON file
    try:
        with open(args.input, 'r') as f:
            full_input_data = json.load(f)
    except FileNotFoundError:
        print(f"Error: Input file not found at {args.input}")
        exit()
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {args.input}")
        exit()
        
    predictions = []
    
    # 3. Iterate through our configured predictors
    for source_name, config in PREDICTOR_CONFIG.items():
        # Check if the data for this source exists in the input file
        if source_name in full_input_data:
            print(f"Found data for '{source_name}'. Running prediction...")
            
            # The actual measurements for this source
            source_data_dict = full_input_data[source_name]
            
            pred_lon, pred_lat = predict_from_source(source_data_dict, config)
            
            if pred_lon is not None:
                predictions.append({
                    'source': source_name,
                    'lon': pred_lon,
                    'lat': pred_lat,
                    'weight': config['fusion_weight']
                })
        else:
            print(f"No data for '{source_name}' in the input file. Skipping.")

    # 4. Fuse the collected predictions
    if not predictions:
        print("Error: Could not generate any predictions from the input data.")
        # Output a default or error coordinate
        final_lon, final_lat = 0.0, 0.0
    else:
        print("\nFusing predictions...")
        fused_lon, fused_lat, total_weight = 0.0, 0.0, 0.0
        for p in predictions:
            print(f"  - From {p['source']}: ({p['lon']:.6f}, {p['lat']:.6f}) with weight {p['weight']:.4f}")
            fused_lon += p['lon'] * p['weight']
            fused_lat += p['lat'] * p['weight']
            total_weight += p['weight']
            
        if total_weight > 0:
            final_lon = fused_lon / total_weight
            final_lat = fused_lat / total_weight
        else:
            # Fallback: if all weights are zero, just take the average of the predictions
            final_lon = np.mean([p['lon'] for p in predictions])
            final_lat = np.mean([p['lat'] for p in predictions])

    # 5. Output the final coordinate in the required format
    print("\n--- FINAL COORDINATE ---")
    print(f"Latitude: {final_lat:.6f}")
    print(f"Longitude: {final_lon:.6f}")
    
    # You might need to write this to a file, e.g., output.txt
    # with open('output.txt', 'w') as f:
    #     f.write(f"{final_lat},{final_lon}\n")